# Task 5 - Gold Layer
# Bygger dimensionella tabeller och views baserat på silver OBT


In [0]:
SILVER_TABLE = "marathos.silver.ultra_marathon_obt"

# Läs från silver
df = spark.table(SILVER_TABLE)
print(f"Rader från silver: {df.count():,}")
display(df.limit(3))

In [0]:
from pyspark.sql.functions import col

dim_event = df.select(
    "event_id",
    "event_name",
    "event_dates",
    "event_distance_length",
    "event_number_of_finishers",
    "year_of_event",
    "distance_unit"
).distinct()

(dim_event.write
          .format("delta")
          .mode("overwrite")
          .saveAsTable("marathos.gold.dim_event"))

print(f"dim_event sparad: {dim_event.count():,} rader")

In [0]:
dim_athlete = df.select(
    col("athlete_id_new").alias("athlete_id"),
    "athlete_country",
    "athlete_gender",
    "athlete_year_of_birth",
    "athlete_age_category",
    "athlete_club"
).distinct()

(dim_athlete.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable("marathos.gold.dim_athlete"))

print(f"dim_athlete sparad: {dim_athlete.count():,} rader")

In [0]:
fct_results = df.select(
    "result_id",
    "event_id",
    col("athlete_id_new").alias("athlete_id"),
    "athlete_performance",
    "performance_value",
    "distance_unit",
    "athlete_average_speed",
    "year_of_event"
)

(fct_results.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable("marathos.gold.fct_results"))

print(f"fct_results sparad: {fct_results.count():,} rader")

In [0]:
spark.sql("""
    CREATE OR REPLACE VIEW marathos.gold.vw_distance_events AS
    SELECT 
        f.result_id,
        f.athlete_id,
        f.athlete_performance,
        f.performance_value,
        f.distance_unit,
        f.athlete_average_speed,
        e.event_name,
        e.event_distance_length,
        e.year_of_event,
        a.athlete_country,
        a.athlete_gender,
        a.athlete_year_of_birth
    FROM marathos.gold.fct_results f
    JOIN marathos.gold.dim_event e ON f.event_id = e.event_id
    JOIN marathos.gold.dim_athlete a ON f.athlete_id = a.athlete_id
    WHERE f.distance_unit IN ('km', 'mi')
""")
print("View vw_distance_events skapad!")

In [0]:
# Skapa view för tidsbaserade event (h)
spark.sql("""
    CREATE OR REPLACE VIEW marathos.gold.vw_timed_events AS
    SELECT 
        f.result_id,
        f.athlete_id,
        f.athlete_performance,
        f.performance_value,
        f.distance_unit,
        f.athlete_average_speed,
        e.event_name,
        e.event_distance_length,
        e.year_of_event,
        a.athlete_country,
        a.athlete_gender,
        a.athlete_year_of_birth
    FROM marathos.gold.fct_results f
    JOIN marathos.gold.dim_event e ON f.event_id = e.event_id
    JOIN marathos.gold.dim_athlete a ON f.athlete_id = a.athlete_id
    WHERE f.distance_unit = 'h'
""")
print("View vw_timed_events skapad!")